## Prepare Climate Time-Series Inputs

**Purpose**
Transforms raw climate series into annual/monthly/daily profiles required by scenario runs.

**Primary inputs**
- `data/source/climate_raw/full.csv`

**Primary outputs**
- `runs/climate_outputs/climatic_data.csv`
- `runs/climate_outputs/climatic_data_daily.csv`
- `runs/climate_outputs/climatic_data_smooth_daily.csv`
- `runs/climate_outputs/climatic_data_month.csv`



## Execution Notes

- Run cells top-to-bottom from a clean kernel.
- Paths are notebook-local and follow the refactored domain layout (`data/` for inputs and small derived tables, `runs/` for generated scenario outputs).
- If you switch datasets or scenarios, update the path/config variables in the first code cells before running downstream steps.



# Climatic data

Input :https://www.renewables.ninja/
Country/Weather data weighted by population.




In [ ]:
import os
import pandas as pd
from project.thermal import TEMP_BASE, ORIENTATION_FACTOR
from project.utils import make_plot


In [ ]:
data = pd.read_csv(os.path.join('data', 'source', 'climate_raw', 'ninja_weather_country_FR_merra-2_population_weighted.csv'), skiprows=[0, 1], index_col=[0], parse_dates=[0])


In [ ]:
data.info()


In [ ]:
data.head()


In [ ]:
make_plot(data.loc[data.index.year == 2006, 'temperature'], 'Temperature (°C)', integer=False, ymin=None)


## Calculation


In [ ]:
df = data.groupby(pd.Grouper(freq='D')).agg({'temperature': 'mean', 'irradiance_surface': 'sum'})

df['smooth_temperature'] = df['temperature'].rolling(7).mean()
df['smooth_irradiance'] = df['irradiance_surface'].rolling(7).mean()

df['heating_season'] = df['temperature'] < TEMP_BASE
df['temperature_season'] = df['temperature'] * df['heating_season']
df['irradiance_season'] = df['irradiance_surface'] * df['heating_season']
variables_output = ['heating_season', 'temperature_season', 'irradiance_season']

df['smooth_heating_season'] = df['smooth_temperature'] < TEMP_BASE
df['smooth_temperature_season'] = df['smooth_temperature'] * df['smooth_heating_season']
df['smooth_irradiance_season'] = df['smooth_irradiance'] * df['smooth_heating_season']
variables_smooth_output = ['smooth_heating_season', 'smooth_temperature_season', 'smooth_irradiance_season']

result_day = df[variables_output].copy()
result_day['irradiance_season'] =  result_day['irradiance_season'] / 1000

result_smooth_day = df[variables_smooth_output].copy()
result_smooth_day['smooth_irradiance_season'] =  result_smooth_day['smooth_irradiance_season'] / 1000

result_month = df[variables_output].groupby(pd.Grouper(freq='M')).sum()
result_month['temperature_season'] =  result_month['temperature_season'] / result_month['heating_season']
result_month['irradiance_season'] =  result_month['irradiance_season'] / 1000

result_year = df[variables_output].groupby(pd.Grouper(freq='Y')).sum()
result_year['temperature_season'] =  result_year['temperature_season'] / result_year['heating_season']
result_year['irradiance_season'] =  result_year['irradiance_season'] / 1000



In [ ]:
result_year.tail()


In [ ]:
var = 'irradiance_season'
for key, factor in ORIENTATION_FACTOR.items():
    result_year['{} {}'.format(var, key)] = result_year[var] * factor
    result_month['{} {}'.format(var, key)] = result_month[var] * factor
    result_day['{} {}'.format(var, key)] = result_day[var] * factor
    result_smooth_day['{} {}'.format(var, key)] = result_day[var] * factor



In [ ]:
# exporting data yearly
export_resirf = result_year[variables_output]
export_resirf.columns = ['DAYS_HEATING_SEASON', 'TEMP_EXT', 'SOLAR_RADIATION']
export_resirf.to_csv(os.path.join('runs', 'climate_outputs', 'climatic_data.csv'))


In [ ]:
export_resirf_daily = result_day[variables_output]
export_resirf_daily.columns = ['DAYS_HEATING_SEASON', 'TEMP_EXT', 'SOLAR_RADIATION']
export_resirf_daily.to_csv(os.path.join('runs', 'climate_outputs', 'climatic_data_daily.csv'))


In [ ]:
export_resirf_smooth_daily = result_smooth_day[variables_smooth_output]
export_resirf_smooth_daily.columns = ['DAYS_HEATING_SEASON', 'TEMP_EXT', 'SOLAR_RADIATION']
export_resirf_smooth_daily.to_csv(os.path.join('runs', 'climate_outputs', 'climatic_data_smooth_daily.csv'))


In [ ]:
# exporting data monthly
export_resirf = result_month[variables_output]
export_resirf.columns = ['DAYS_HEATING_SEASON', 'TEMP_EXT', 'SOLAR_RADIATION']
export_resirf.to_csv(os.path.join('runs', 'climate_outputs', 'climatic_data_month.csv'))


### Climatic data


In [ ]:
temp_2006 = {'raw': result_day.loc[result_day.index.year == 2006, 'temperature_season'],
 'smooth': result_smooth_day.loc[result_day.index.year == 2006, 'smooth_temperature_season']
 }
make_plot(pd.DataFrame(temp_2006), 'Temperature (°C)', integer=False, ymin=None)
